# CircuitSage — Stack traces for circuits.

**Tagline.** *An offline Gemma-4-powered lab partner that turns silent bench failures into a step-by-step debugging path.*

**Tracks.** Future of Education (primary) · Digital Equity & Inclusivity (offline microserver) · Safety & Trust (uncertainty calibration + low-voltage refusal).

**Stretch prize.** Unsloth Prize — fine-tuned LoRA on a synthetic circuit-fault Q&A dataset, served via Ollama.

**Demo bench.** Inverting op-amp amplifier, expected gain −4.7, observed output stuck near +12 V. The fault: floating non-inverting input. CircuitSage asks for the missing measurement, then identifies the reference fault and explains why the gain equation was not the immediate problem.

Repo: https://github.com/<org>/circuitsage   ·   Dataset: https://www.kaggle.com/datasets/karansinghbisht/circuitsage-faults-v1   ·   Training kernel: https://www.kaggle.com/code/karansinghbisht/circuitsage-gemma-lora

## 1. Problem

Software students get stack traces. Electronics students get silence. When a circuit fails, the symptom is a flat oscilloscope trace, a saturated op-amp output, a hot part, or no signal at all. The student has to connect theory, simulation, breadboard wiring, datasheets, and instrument readings while one instructor helps ten other people. Many labs stop being educational and become trial-and-error.

CircuitSage is a local-first Gemma 4 lab partner for low-voltage educational circuits. It does not replace an instructor. It makes the debugging process explicit: what evidence exists, what is expected, what was observed, which safety rules apply, and which **single** measurement should happen next.

## 2. Solution

Four surfaces, one persistent session:

- **Studio** — desktop workspace for artifacts, tool calls, diagnosis, and reports (`/studio/:id`).
- **Bench Mode** — phone-friendly measurement and photo capture, paired by QR (`/bench/:id`).
- **Companion** — persistent screen-aware helper for LTspice / Tinkercad / MATLAB / oscilloscope screenshots (`/companion`).
- **Educator** — class-wide dashboard: repeated faults, safety refusals, unresolved sessions, stalled measurements (`/educator`).

Every diagnosis is a structured JSON: `experiment_type`, `expected_behavior`, `observed_behavior`, ranked `likely_faults` with confidence, the `next_measurement` to take, a `safety` block, a plain-language `student_explanation`, and a `confidence` enum. The same schema flows through Bench, Studio, Companion, the iOS app, and the PDF report.

## 3. Architecture

```
 iOS Bench (camera + voice + on-device Gemma 3n)
        |
        v
 Web Studio (PWA)  <----  Desktop Companion (Electron, screen watcher)
        |
        v
 FastAPI backend
   - Safety check
   - SPICE topology parser (R/C/L/V/I/D/Q/M/X/E)
   - 13 fault catalogs (39 fault entries, signature-based scoring)
   - Vision tool (Gemma 4 vision)
   - Vector RAG (Chroma + nomic-embed-text 768d)
   - Iterative agentic loop (native Gemma 4 function calling)
   - PDF report generator (ReportLab)
        |
        v
 Ollama local: gemma4:e4b   +   circuitsage:latest (Unsloth LoRA, optional)
```

The whole stack runs on a single laptop or microserver with no internet. The hosted demo uses the same container with rate-limiting and seeded data.

## 4. Multimodal

- **Vision in the main flow** (not just companion): `breadboard` and `oscilloscope` artifact kinds run through `tools/vision.py` and feed their structured output into the diagnosis context. Two images max per diagnosis.
- **Schematic → netlist** via Gemma 4 vision: a hand-drawn or photographed schematic is converted to a SPICE-style netlist, validated by `parse_netlist_text`, and rejected with `mode: vision_unavailable` and `confidence: 0.0` rather than fabricated when the photo is ambiguous.
- **Companion**: persistent screen-share watcher that captures frames from LTspice / Tinkercad / MATLAB / browser labs, runs a two-step Gemma 4 vision-then-structure call, and returns workspace-specific next actions.
- **Datasheet auto-detect**: parsed model strings (`2N3904`, `1N4148`, `TL072`) trigger an automatic `lookup_datasheet` tool call that injects pin map + abs max + common faults into the LLM context.

## 5. Fine-tune (Unsloth LoRA)

Dataset: 6,000 chat-format examples (`{"messages": [system, user, assistant]}`), 4,978 unique user prompts, max-duplicate 4. 13 topology fault classes plus a 5 % safety-refusal branch. The assistant always emits the structured-diagnosis JSON.

Distribution lint passes via `train/dataset/validate.py`:
- per-topology in [10 %, 30 %]
- safety-refusal rows in [4 %, 6 %]
- negative-evidence rows in [4 %, 10 %]
- ≥ 1500 unique user prompts

Training: Unsloth + TRL `SFTTrainer`, 4-bit base `unsloth/gemma-3-1b-it-bnb-4bit`, LoRA r=16, alpha=32, lr 2e-4, 3 epochs, seed 3407. Saves the adapter and a q4_k_m GGUF. Loaded into Ollama as `circuitsage:latest` via `train/output/circuitsage.Modelfile` so the local backend can pick it up automatically.

## 6. Eval harness

200-example deterministic holdout (every 30th dataset row). `train/eval/harness.py --model <ollama-tag>` records:

- `schema_validity_rate` — JSON validity + required keys + valid `confidence` enum
- `experiment_type_exact_match`
- `top_fault_id_match`
- `safety_refusal_precision` and `safety_refusal_recall`
- `mean_latency_ms`

Results land in `train/eval/last_run.json` and feed the model card.

In [ ]:
# Sample structured diagnosis (deterministic catalog path with Ollama down)
import json
DEMO = {
  "experiment_type": "op_amp_inverting",
  "expected_behavior": {"gain": -4.7, "output": "inverted sine wave, about 4.7 V peak for 1 V peak input"},
  "observed_behavior": {"summary": "Output is stuck near the positive supply rail",
                          "evidence": ["Vout = 11.8 V DC", "V+ = 12.1 V DC", "V- = -12.0 V DC",
                                       "comparison: saturation_instead_of_linear_amplification"]},
  "likely_faults": [
    {"id": "floating_noninv_input", "fault": "Floating non-inverting input", "confidence": 0.86,
     "why": "An inverting op-amp needs the non-inverting input tied to circuit ground; a floating reference can drag the output to a rail."},
    {"id": "missing_feedback", "fault": "Open or wrong feedback path", "confidence": 0.55,
     "why": "Without negative feedback, the op-amp goes open-loop and saturates."},
    {"id": "rail_imbalance", "fault": "Power rail or ground problem", "confidence": 0.35,
     "why": "A missing or unequal rail can clamp the output."}
  ],
  "next_measurement": {"label": "Voltage at non-inverting input pin", "expected": "approximately 0 V",
                        "instruction": "Measure the non-inverting input with respect to circuit ground before changing resistor values."},
  "safety": {"risk_level": "low_voltage_lab", "warnings": ["Power off before rewiring the op-amp pins."]},
  "student_explanation": "The simulation expects gain around -4.7, but the observed output is saturated near the positive rail. Both supply rails look correct, so the next best measurement is the non-inverting input voltage.",
  "confidence": "medium_high",
  "session_status": "diagnosing",
  "gemma_status": "deterministic_fallback",
  "gemma_model": "gemma4:e4b"
}
print(json.dumps(DEMO, indent=2))

## 7. Reproducibility

```bash
make install        # python venv + npm + ollama pull gemma4:e4b
make demo-seed      # 8 educator-ready sessions
bash scripts/demo_smoke.sh    # hits all 13 seed endpoints + PDF + educator overview
make demo           # backend + frontend dev server
```

CI: backend pytest (103 tests), frontend Vite build, desktop check, dataset validate, i18n parity check.

## 8. Safety and uncertainty

Two layers:

1. **Refusal** — mains, SMPS primary, CRT/flyback, EV battery, microwave HV cap, large-bank capacitors. The diagnosis returns `experiment_type: safety_refusal` with explicit warnings.
2. **Calibration** — the **uncertainty gallery** has eight cases where the right answer is to ask for more evidence: op-amp netlist missing rails, RC filter missing capacitor, free-form question without artifacts, conflicting repeated measurements, breadboard photo without schematic, unsupported Wien bridge, voltage reading entered as resistance, low-voltage wall-wart edge case. In all eight, the diagnosis returns `confidence: low` and `next_measurement.label != "Stop live debugging"`. A regression test (`backend/tests/test_uncertainty_gallery.py`) parametrizes over the eight cases.

## 9. What's actually working today

- 103 backend tests pass (Group A–F hardening + integration + uncertainty + datasheet auto-detect + agent loop).
- Frontend production build is clean (Vite 6, framer-motion, wouter routing, four locales en/hi/es/pt, accessibility prefs, theme toggle, onboarding tour).
- 13 topology seed endpoints all return non-canned diagnoses grounded in the fault catalog.
- Vector RAG with Chroma + 768-dim Ollama nomic-embed-text returns the correct fault doc as top hit for `"floating non-inverting input"`.
- PDF report endpoint returns a 3-page `application/pdf` with `%PDF` magic bytes (`backend/tests/test_report_pdf.py`).
- Hosted demo packaging: Dockerfile (multi-stage frontend builder + Python runtime + Ollama), `fly.toml` for deploy.
- Dataset upload + training kernel pushed to Kaggle. The training kernel runs on Kaggle's free T4 GPU (Unsloth + bitsandbytes pinned for SM 7.5 compatibility).
- Live agent loop tested with `gemma3:4b` (single-shot, tools-fallback path) and `gemma4:e4b` (native function calling).
- Honest `gemma_status` enum: `ollama_gemma_agentic`, `ollama_gemma_single_shot`, `ollama_partial`, `deterministic_fallback`, `blocked_by_safety`.

USER ACTION items remaining: physical iPhone airplane-mode test, Hugging Face publish (cards prepared), Fly deploy, GitHub repo public, demo video shoot, Kaggle final submission.

## 10. Limitations

- The schematic-to-netlist Gemma 4 vision pass is brittle on noisy photos. CircuitSage now returns `confidence: 0.0` and `mode: vision_unavailable` rather than fabricating a netlist.
- Dataset diversity is synthetic. The augmentation step uses Ollama-Gemma paraphrases when reachable and falls back to rule-based variants when not (logged in `train/dataset/build.py`).
- On-device iOS uses an `Engine` interface ready for Cactus or MediaPipe LLM Inference. Bundling the actual Gemma E2B `.task` file is a USER ACTION pending license acceptance.
- The eval harness depends on Ollama being reachable; the holdout set is committed but `last_run.json` will be regenerated each time the harness runs.

## 11. Future work

Real LTspice/MATLAB importers, richer schematic recognition (curated training photos), physical iPhone airplane-mode demo with Cactus, larger real-student dataset, expanded uncertainty benchmark, hosted Educator analytics with anonymous opt-in telemetry, and Hindi/Spanish/Portuguese voice models for the offline microserver framing.